<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/K21_Konsult_52.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — Interpretive Jyotiṣa

**Khona21** is an experimental AI-assisted Jyotiṣa interpretation platform, developed as a companion to **Parashar21**, which performs the astronomical calculations underlying traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform combines precomputed Rashi and Navamsha charts, Vimshottari Dasha schedules and planetary transits (Gochar) with a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on hard-coded interpretive rules, Khona21 employs **source-grounded, multipass LLM reasoning** to examine planetary configurations, yogas, cancellations, exceptions and conflicting interpretations. An initial assessment establishes the horoscope's principal characteristics, while subsequent consultations investigate specific questions using the original reference texts and relevant astrological evidence.

This notebook, **k21_KhonaPlus**, implements the consultation workflow, integrating deterministic calculations from Parashar21 with the interpretive capabilities of modern Large Language Models.

**Khona21 explores the convergence of traditional Jyotiṣa knowledge and modern artificial intelligence, combining computational precision with source-grounded reasoning to produce systematic, transparent and auditable astrological interpretations.**

In [1]:
from datetime import datetime
import pytz
print("\033[1m"+'জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।'+"\033[0m")
print(datetime.now(pytz.timezone('Asia/Calcutta')))

জন্ম-মৃত্যু-বিবাহ, তিন না জানেন বরাহ।
2026-11-04 11:20:23.446989+05:30


#Setup, Installs, Imports

In [2]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown


!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
from k21core import *
from k21fileops import *

In [3]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
#import k21core
#import importlib

#importlib.reload(k21core)
from k21core import *

In [ ]:
# ------------------------------------------------
# Use Claude
# ------------------------------------------------
#!pip install -q anthropic

#provider = "claude"
#model = "claude-haiku-5-5"


In [4]:
# ------------------------------------------------
# Use OpenAI
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
#model = "gpt-6-luna"
model = "gpt-6.1-sol"
#model = "gpt-6-astra"

In [36]:
# ------------------------------------------------
# Set Native ID
# ------------------------------------------------
Native_id = "TopGun"

###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [5]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


In [6]:
# ------------------------------------------------
# Locate necessary file and set up paths to the same
# ------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
#ls /content/drive/MyDrive/Prithwis/Parashar21/K21

K21_HOME = "/content/drive/MyDrive/Prithwis/Parashar21/K21"

# ------------------------------------------------
# The Corpus Files, Braha and Harihar, are available
# in GitHub Repo and can be downloaded with wget
# ------------------------------------------------
BRAHA   = f"{K21_HOME}/Corpus/BR_Curated_22.txt"
HARIHAR = f"{K21_HOME}/Corpus/HM_Clean_07.txt"
RAMAN = f"{K21_HOME}/Corpus/Raman_08.md"
YOGS = f"{K21_HOME}/Corpus/300_Astrology_Yogas.md"



Mounted at /content/drive


## Load Files

In [ ]:
#!wget -O braha.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/BR_Curated_22.txt"
#!wget -O harihar.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/HM_Clean_07.txt"
#!wget -O role.txt -q https://raw.githubusercontent.com/prithwis/parashar21/main/RoleFiles/khona261029A.txt

#BRAHA   = "/content/braha.txt"
#HARIHAR = "/content/harihar.txt"
#ROLE    = "/content/role.txt"

In [7]:
#
# Following files from Parashar21 need to be uploaded here
#

import zipfile
from google.colab import files

# Upload ZIP file
uploaded = files.upload()
zipName = next(iter(uploaded))

# Extract all files into current directory
with zipfile.ZipFile(zipName, 'r') as zf:
    zf.extractall()

print("Extracted:", zipName)

CHART = "/content/Chart.txt"
NAVAMSA = "/content/Navamsa_Chart.txt"
DASHA = "/content/Dasha.txt"
GOCHAR = "/content/Gochar_Chart.txt"
GOCHAR_EVENTS = "/content/Transit.txt"

Saving Kompute_261104_055401.zip to Kompute_261104_055401.zip
Extracted: Kompute_261104_055401.zip


# LLM Access




In [38]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
ROLE1 = f"{K21_HOME}/Roles/ramanyog261104A.txt"
raman = readText(ROLE1)
ROLE2 = f"{K21_HOME}/Roles/khona261104A.txt"
khona21 = readText(ROLE2)

/content/drive/MyDrive/Prithwis/Parashar21/K21/Roles/ramanyog261104A.txt | 4,533 characters
/content/drive/MyDrive/Prithwis/Parashar21/K21/Roles/khona261104A.txt | 4,668 characters


## 300 YOGS

In [18]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)
'''
BaseCONTEXT = getContext(
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA
)
'''

CONTEXT = getContext(
    RAMANYOG=YOGS,
    CHART=CHART
)


In [19]:
len(CONTEXT)

188154

In [11]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
n_tokens = len(enc.encode(CONTEXT))
print(f"Context tokens: {n_tokens:,}")

Context tokens: 40,671


In [21]:
YOG_INSTRUCTION = """
Examine all yoga definitions in YOGS against CHART.
Identify and verify applicable yogas, documenting their formation
conditions, chart evidence and stated outcomes.
Report uncertain formations separately.
"""

In [22]:
# ------------------------------------------------
# Assemble the yog query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, YOG_INSTRUCTION)

RamanYog = callLLM(
    client=client,
    role=raman,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6.1-sol | Effort: medium | In: 41,609 | Cached: 0 | Out: 10,535 | Reasoning: 6,214 | Time: 229.2s


In [23]:
# ------------------------------------------------
# Save RamanYog as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(model, RamanYog)
display(Markdown(RamanYog))

Saved: /content/K21_261104_120845_gpt-6.1-sol.md


# RamanYog Verification Report

Source-entry numbers below follow the order of the **300 supplied definitions**, preserving repeated names as distinct entries.

**Method:** Planetary positions, conjunctions, aspects and dignities are taken from CHART. Where a formation depends on an unspecified natural-versus-functional benefic/malefic convention, that uncertainty is retained rather than silently resolved. Stated outcomes are source attributions—not guarantees, medical diagnoses or findings about character.

## 1. A. VERIFIED YOGAS

### 1.1 Sunapha Yoga — Entry 2

- **Formation:** “Any planets, except Sun, in the second house from the Moon.”
- **Chart evidence:** Moon occupies Sagittarius. Jupiter and Saturn occupy Capricorn, second from Moon. Both satisfy the stated exclusion of Sun.
- **Stated outcomes:** Self-earned property, substantial wealth, intelligence, sound decisions and ruler-like reputation.
- **Qualifications:** The supplied rule adds no strength or dignity requirement. Jupiter’s debilitation therefore does not prevent formation under this entry.

### 1.2 Sakata Yoga — Entry 12

- **Formation:** “Moon in 12th, 6th or 8th house from Jupiter.”
- **Chart evidence:** Jupiter occupies Capricorn; Moon occupies Sagittarius, twelfth from Jupiter.
- **Stated outcomes:** Loss and subsequent recovery of fortune, hardship, poverty or privation, stubbornness and difficulties with relatives.
- **Qualifications:** These are the source’s attributed outcomes; formation alone does not establish their occurrence.

### 1.3 Vesi Yoga — Entry 16

- **Formation:** “Planets other than Moon occupy 2nd position from Sun.”
- **Chart evidence:** Sun occupies Virgo. Mars and Mercury occupy Libra, second from Sun.
- **Stated outcomes:** Fortune, happiness, ethical disposition, fame and aristocratic standing.
- **Qualifications:** No planetary-strength requirement appears in this definition.

### 1.4 Sasa Yoga — Entry 21

- **Formation:** “Saturn occupies a Kendra of his own house or exaltation sign.”
- **Chart evidence:** Saturn occupies Bhava 7, a kendra, in Capricorn. CHART explicitly identifies Saturn as occupying its own house.
- **Stated outcomes:** Command over subordinates, leadership or rulership, alongside questionable conduct, covetousness and an extreme temperament.
- **Qualifications:** Own-sign placement establishes the dignity requirement; exaltation is not necessary.

### 1.5 Sareera Soukhya Yoga — Entry 108

- **Formation:** “The lord of Lagna, Jupiter or Venus should occupy a quadrant.”
- **Chart evidence:** Jupiter occupies Bhava 7, satisfying the Jupiter alternative independently.
- **Stated outcomes:** Longevity, freedom from major disease, wealth, prosperity and interest in politics.
- **Qualifications:** Jupiter is debilitated, but this formation clause does not require strength or favorable dignity. Its health claims are not medical conclusions.

### 1.6 Swaveeryaddhana Yoga — Entry 132

- **Formation:** “The 2nd lord should occupy a quadrant or trine from the 1st lord or the 2nd lord being a bene fic should be in either a deep exaltation or conjunction with an exalted planet.”
- **Chart evidence:** Second lord Sun occupies Virgo; first lord Moon occupies Sagittarius. Virgo is tenth from Sagittarius, satisfying the first alternative.
- **Stated outcomes:** Prosperity obtained through personal effort and hardship, potentially including building one’s own business.
- **Qualifications:** The independently satisfied first alternative requires neither benefic classification nor exaltation.

### 1.7 Bandhu Pujya Yoga — Entry 194

- **Formation:** “The fourth house or the fourth Lord has the association or aspect of Jupiter.”
- **Chart evidence:** Fourth lord Venus is explicitly aspected by Jupiter.
- **Stated outcomes:** Good reputation among family and friends, affection, trust and appreciation of empathetic conduct. This entry specifically attributes being more loved than revered.
- **Qualifications:** Jupiter’s debilitation does not invalidate the stated aspect-based formation.

### 1.8 Bandhubhisthyaktha Yoga — Entry 195

- **Formation:** “The fourth Lord is associated with malefics or occupies evil shashtiamsas or joins inimical or debilitation signs.”
- **Chart evidence:** Fourth lord Venus occupies Virgo and is explicitly marked debilitated.
- **Stated outcomes:** Being misunderstood, mistreated or deserted by relatives and associates.
- **Qualifications:** The debilitation alternative is sufficient; malefic classification and Shashtiamsa data are unnecessary for this verification.

### 1.9 Nishkapata Yoga — Entry 205

- **Formation:** “The fourth house is occupied by a benefic, or a planet in exaltation, friendly or own house, or the fourth house is a benefic sign.”
- **Chart evidence:** Mercury occupies Bhava 4 and is explicitly marked “In friendly house.”
- **Stated outcomes:** Sincerity, a loving disposition, respect for others and dislike of hypocrisy, secrecy and betrayal.
- **Qualifications:** Verification uses the friendly-house alternative, not an assumed benefic classification of Mercury.

### 1.10 Ekaputra Yoga — Entry 225

- **Formation:** “Lord of the fifth house joins a kendra or trikona.”
- **Chart evidence:** Fifth lord Mars occupies Bhava 4, a kendra.
- **Stated outcomes:** One child, described as most likely a son; the source also discusses limitations on further children.
- **Qualifications:** Formation does not establish an actual number, sex or survival outcome for children.

### 1.11 Satkalatra Yoga — Entry 239

- **Formation:** “The Lord of the seventh house or Venus is joined or is aspected by Jupiter or Mercury.”
- **Chart evidence:** Seventh lord Saturn is explicitly conjunct Jupiter.
- **Stated outcomes:** A noble, virtuous, disciplined and supportive spouse.
- **Qualifications:** The seventh-lord–Jupiter conjunction independently satisfies the rule; no benefic-status substitution is needed.

### 1.12 Raja Yoga — Entry 250

- **Formation:** “The Lord of the sign, a planet is debilitated in, or the planet, that would be exalted there, is in a kendra from the Moon or Lagna.”
- **Chart evidence:**
  - Jupiter is debilitated in Capricorn. Capricorn’s lord Saturn occupies Bhava 7, a kendra from Lagna.
  - Venus is debilitated in Virgo. Virgo’s lord Mercury occupies Bhava 4, also a kendra from Lagna.
- **Stated outcomes:** Ruler-like standing, influence, respect, power and substantial wealth.
- **Qualifications:** Both instances satisfy this **single source entry**. This establishes its stated formation, not an additional, independently imported cancellation doctrine.

## 2. B. CONDITIONAL YOGAS

### 2.1 Vasumathi Yoga — Entry 9

- **Formation:** “Benefics occupy the upachayas 3, 6, 10, or 11 either from the ascendant or from the Moon.”
- **Possible match:** Venus is tenth and Mercury eleventh from Moon; Venus is also third from Lagna. Moon, explicitly a functional benefic, occupies sixth from Lagna.
- **Unresolved condition:** The source does not specify natural versus functional benefics, or whether some benefics suffice. Jupiter is not in an eligible upachaya from either reference, so an all-natural-benefics reading would not establish this formation.

### 2.2 Amala Yoga — Entry 13

- **Formation:** “10th from Moon or Lagna should be occupied by any benefic planet”
- **Possible match:** Venus occupies Virgo, tenth from Moon.
- **Unresolved condition:** Venus’s required benefic classification is not supplied by CHART; the source does not identify which benefic convention applies.

### 2.3 Parvata Yoga — Entry 14

- **Formation:** “6th or 8th houses should be either unoccupied or occupied by benefic planets.”
- **Possible match:** Sixth house contains Moon, explicitly a functional benefic.
- **Unresolved condition:** Verification depends on accepting functional beneficence and reading “or” as allowing the sixth house alone to suffice. Eighth house contains Ketu, whose relevant classification is not supplied.

### 2.4 Vasi Yoga — Entry 17

- **Formation:** “Planets other than Moon occupy 12th position from Sun.”
- **Possible match:** Rahu occupies Leo, twelfth from Sun in Virgo.
- **Unresolved condition:** The rule does not specify whether its planetary scope includes Rahu and Ketu. This match depends entirely on including Rahu.

### 2.5 Obhayachari Yoga — Entry 18

- **Formation:** “Planets other than Moon are on either side of the Sun.”
- **Possible match:** Rahu occupies the sign immediately preceding Sun; Mars and Mercury occupy the sign immediately following Sun.
- **Unresolved condition:** Rahu must count as an eligible planet for the preceding-side condition.

### 2.6 Parijatha Yoga — Entry 47

- **Formation:** “Lord of the sign in which the lord of the house occupied by the Ascendant lord, or the lord of Navamasa occupied by the lord of Rasi in which the Ascendant lord is positioned joins a quadrant, a trine or his own exaltation signs.”
- **Possible match:** Lagna lord Moon occupies Sagittarius, whose lord is Jupiter. Jupiter occupies Capricorn, whose lord Saturn occupies seventh house.
- **Unresolved condition:** This matches a successive-dispositor reading of the first alternative, but that alternative has an incomplete placement relationship in its wording. It cannot be verified without resolving that omission.

### 2.7 Brahma Yoga — Entry 63

- **Formation:** “Jupiter and Venus are in Kendras respectively from the lords of the 9th and 11th, and Mercury is in a similar position from the lord of either Lagna or the 10th.”
- **Possible match:** Ninth lord is Jupiter itself; eleventh lord is Venus itself. Mercury is conjunct tenth lord Mars.
- **Unresolved condition:** The first two relationships require counting each planet as occupying a kendra from itself. The source does not establish that this self-reference is intended.

### 2.8 Ardha Chandra Yoga — Entry 79

- **Formation:** “All planets occupy the seven houses beginning from a Panapara or Apoklima.”
- **Possible match:** All nine listed planetary bodies fall within the seven-house arc from Bhava 2 through Bhava 8.
- **Unresolved condition:** The source does not define Panapara/Apoklima, or clarify whether confinement within seven houses suffices rather than occupation of every house in that sequence.

### 2.9 Damni Yoga — Entry 92

- **Formation:** “All planets must occupy any six signs.”
- **Possible match:** Including Rahu and Ketu, the planets occupy six signs: Leo, Virgo, Libra, Sagittarius, Capricorn and Aquarius.
- **Unresolved condition:** “All planets” must include the nodes. The seven non-node planets occupy only four signs.

### 2.10 Kedara Yoga — Entry 94

- **Formation:** “All planets must occupy any four signs.”
- **Possible match:** The seven non-node planets occupy Virgo, Libra, Sagittarius and Capricorn.
- **Unresolved condition:** “All planets” must exclude the nodes. Including them produces six occupied signs.
- **Qualification:** Entries 92 and 94 remain distinct; their possible matches depend on different planetary scopes.

### 2.11 Sarpa Yoga — Entry 102

- **Formation:** “All the malefics occupy kendras.”
- **Possible match:** Every planet explicitly labeled a functional malefic—Mercury, Jupiter and Saturn—occupies a kendra.
- **Unresolved condition:** The source must intend this supplied functional-malefic set. CHART does not supply a complete natural-malefic classification or functional labels for every body.

### 2.12 Sada Sanchara Yoga — Entry 117

- **Formation:** “The lord of either the Lagna or the sign occupied by Lagna lord must be a movable sign.”
- **Possible match:** Moon occupies Sagittarius; its sign lord Jupiter occupies Capricorn, a movable sign.
- **Unresolved condition:** The source says a lord “must be a movable sign,” rather than occupy one. The possible match requires that interpretive repair.

### 2.13 Matrumooladdhana Yoga — Entry 138

- **Formation:** “The lord of the 2nd joins the 4th lord or is aspected by him the above yoga.”
- **Possible match:** Second lord Sun is conjunct fourth lord Venus.
- **Unresolved condition:** The conjunction clause is satisfied, but the dangling phrase “the above yoga” leaves uncertain whether another formation or prerequisite was intended.

### 2.14 Daridra Yoga — Entry 150

- **Formation:** “The lord of the 5th joins with the lord of 6th, 8th or 12th without beneficial aspects or conjunctions.”
- **Possible match:** Fifth lord Mars is conjunct twelfth lord Mercury. Mars receives Saturn’s aspect. Mercury and Saturn are explicitly functional malefics.
- **Unresolved condition:** Absence of beneficial influence depends on the classification intended for Mercury and the meaning of “beneficial.” A functional-only reading supports the match; a different convention is not established.

### 2.15 Bhratruvriddhi Yoga — Entry 177

- **Formation:** “The third Lord, or Mars, or the third house are joined or aspected by benefics or strong.”
- **Possible match:** Third lord Mercury is conjunct Mars, explicitly a functional benefic. Jupiter also aspects the third house.
- **Unresolved condition:** The first route depends on accepting functional beneficence. Jupiter’s relevant benefic status and the alternative strength condition are not established.

### 2.16 Kapata Yoga — Entry 202

- **Formation:** “The fourth house is joined by a malefic and the fourth Lord is associated with or aspected by malefics or is hemmed in between malefic.”
- **Possible match:** Functional malefic Mercury occupies fourth house. Fourth lord Venus receives the aspect of functional malefic Jupiter.
- **Unresolved condition:** Both requirements are satisfied under the supplied functional labels, but the source does not establish whether those, rather than natural classifications, are intended.

## 3. C. UNVERIFIABLE YOGAS

Each item below identifies a distinct source entry.

1. **Vanchana Chora Bheethi Yoga — Entry 11:** The alternative involving Gulika’s association with kendra/trikona lords cannot be checked because Gulika is absent from CHART.

2. **Sankha Yoga — Entry 44:** Fifth lord Mars and sixth lord Jupiter occupy mutual kendras. The required powerful disposition of Lagna lord Moon is not supplied.

3. **Garuda Yoga — Entry 66:** Moon’s Navamsa lord and its exaltation status, daytime birth and the required waxing status are not supplied as chart attributes.

4. **Krisanga Yoga — Entry 112:** Neither “dry sign” nor “dry planet” is defined in the supplied references. Cancer’s water classification does not resolve those separate categories.

5. **Swaveeryaddhana Yoga — Entry 131:** The nested Navamsa-dispositor chain and its required strength cannot be established without divisional and strength data.

6. **Netranasa Yoga — Entry 164:** The independent “or they are in Neechamsa” alternative cannot be tested without the relevant divisional placements; its pronoun reference is also unclear.

7. **Durmukha Yoga — Entry 168:** Rahu occupies second house and second lord Sun joins Venus, but the required malefic/“evil planet” classifications are not established.

8. **Durmukha Yoga — Entry 169:** Verification requires an established “evil” classification of second lord Sun and either Gulika data or the specified unfavorable Navamsa conditions.

9. **Bhojana Soukhya Yoga — Entry 170:** Jupiter aspects second lord Sun, but Sun’s required powerful status and Vaiseshikamsa are unavailable.

10. **Parannabhojana Yoga — Entry 172:** Debilitated Jupiter aspects second lord Sun. Sun’s possible placement in an unfriendly Navamsa cannot be checked.

11. **Sarpaganda Yoga — Entry 174:** Rahu occupies second house; Mandi’s position is missing.

12. **Vakchalana Yoga — Entry 175:** Second-lord malefic status and cruel Navamsa placement are not established. These are essential requirements.

13. **VishapraYoga Yoga — Entry 176:** Second house contains Rahu and receives Ketu’s aspect; second lord Sun receives debilitated Jupiter’s aspect. Required malefic classifications and Sun’s cruel Navamsa remain unestablished.

14. **Parakrama Yoga — Entry 182:** Third lord Mercury joins functional benefic Mars, but Mercury’s benefic Navamsa and the required “benefic signs” status of Mars’s placement are unavailable.

15. **Yuddha Praveena Yoga — Entry 183:** The nested Navamsa-lord relationships and “own vargas” condition require unavailable divisional data.

16. **Satkathadisravana Yoga — Entry 186:** Jupiter aspects third house, but the required benefic-sign, benefic-aspect and benefic-amsa classifications are not established.

17. **Matrunasa Yoga — Entry 199:** The successive Navamsa-dispositor chain beginning with fourth lord Venus cannot be calculated from the supplied chart.

18. **Nishkapata Yoga — Entry 206:** The possible Parvata/Uttamamsa alternative requires unavailable specialized amsa data. Its independence from the fourth-house requirement is also unclear.

19. **Matru Sneha Yoga — Entry 208:** The natural or temporal friendship alternative for Moon and Venus cannot be checked from the supplied relationship data.

20. **Anapathya Yoga — Entry 211:** The required weakness of Jupiter and the first, seventh and fifth lords is not supplied. Jupiter’s debilitation alone cannot establish every required weakness.

21. **Kalanirdesat Putranasa Yoga — Entry 230:** The formation, “Malefics are Jupiter and Lagna,” does not express an intelligible positional relationship.

22. **Theevrabuddhi Yoga — Entry 232:** Fifth lord Mars is a functional benefic, but its Navamsa lord and that lord’s benefic aspects are unavailable; the intended benefic convention is also unspecified.

23. **Thrikalagnana Yoga — Entry 234:** Mrudwamsa/Gopuramsa and Jupiter’s required Navamsa status are unavailable.

24. **Bhagya Yoga — Entry 241:** Venus occupies third house and aspects ninth house, but its required strong-benefic status is not established.

25. **Raja Yoga — Entry 246:** Jupiter and Venus are debilitated, but “bright rays” and “favourable positions” are undefined or unavailable for the applicable route.

26. **Raja Yoga — Entry 247:** Digbala values or explicit Digbala classifications are not supplied.

27. **Raja Yoga — Entry 252:** Jupiter and Venus supply debilitated Rashi candidates, but their required exalted Navamsa placements are unavailable.

28. **Raja Yoga — Entry 258:** Moon’s Navamsa lord is unavailable, preventing verification of its quadrant/trine relationship to Lagna or Mercury.

29. **Guhyaroga Yoga — Entry 284:** Moon’s Cancer/Scorpio Navamsa placement and the required malefic association there cannot be checked.

30. **Khalwata Yoga — Entry 295:** Saturn aspects Cancer Lagna, but the source’s separate “malefic sign” classification is not defined. Cancer’s water/movable classifications do not resolve it.

31. **Rajabhrashta Yoga — Entry 297:** Aroodha Lagna, Aroodha Dwadasa and their lords are not supplied.

## 4. D. AUDIT SUMMARY

- **Source entries examined:** 300
- **VERIFIED:** 12
- **CONDITIONAL:** 16
- **UNVERIFIABLE:** 31

Repeated names and alternative formation clauses were screened separately. No Dasha, transit, event-timing or general life interpretation is included.

## First Pass

In [24]:
# ------------------------------------------------
# Set the Context for FirstPass
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)

CONTEXT = getContext(
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA
)



In [25]:
len(CONTEXT)

944654

In [27]:
# ------------------------------------------------
# FIRST PASS
#
# ------------------------------------------------

FirstPass_Instruction = """
FIRST PASS — FOUNDATIONAL ASSESSMENT

Prepare a concise, high-level assessment of this horoscope.

Present an overall picture of the native's principal
strengths, weaknesses and significant planetary configurations.

Identify the most important favourable and unfavourable yogas,
sign exchanges, cancellations and mitigating factors,
to the extent that they are evident from the supplied data
and reference texts.

Begin with a simple-language summary, followed by a brief
astrological explanation with relevant source references.

This is an initial orientation, not an exhaustive audit.
Focus on the dominant features of the horoscope rather than
attempting to catalogue every planetary condition or yoga.

Distinguish chart facts from interpretations, and acknowledge
important contradictions or uncertainties.

Do not invent configurations or use rules unsupported by
the supplied reference texts.

Do not make predictions about specific future events.
"""

In [28]:
# ------------------------------------------------
# Assemble the first pass query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, FirstPass_Instruction)

FirstPass = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)


OPENAI | gpt-6.1-sol | Effort: medium | In: 222,171 | Cached: 0 | Out: 2,181 | Reasoning: 588 | Time: 67.9s


In [29]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(model, FirstPass)
display(Markdown(FirstPass))

Saved: /content/K21_261104_122621_gpt-6.1-sol.md


## 1. Simple-language summary

This horoscope combines **capacity for sustained effort, technical or organisational work, and leadership** with significant pressures around **peace of mind, home life, partnerships, and financial security**.

Its principal support is Mars, which rules both achievement and the fifth house of intelligence and merit. Saturn is also prominent and strongly placed by sign, suggesting endurance and responsibility—but its strength does not make all its effects comfortable. Jupiter and Venus are debilitated, although both receive important cancellation or mitigating conditions.

The overall picture is therefore **mixed rather than uniformly fortunate or unfortunate**: useful abilities and opportunities coexist with obstacles, demanding relationships, and difficulty translating accomplishment into contentment. This is an interpretation within the supplied astrological framework, not a prediction of specific events.

## 2. Principal favourable configurations

- **Mars is the principal functional benefic.**  
  **Chart fact:** For Cancer ascendant, Mars rules houses 5 and 10 and occupies the angular fourth house, aspecting its own tenth house and the eleventh.  
  **Interpretation:** This supports initiative, applied intelligence, professional drive, and the ability to build tangible results. Both Braha’s *Raja Yogakarakas* and Harihar’s Cancer-ascendant discussion identify Mars as a raja-yogakaraka. Its afflictions, discussed below, qualify rather than erase this support.

- **Saturn has substantial sign and divisional support.**  
  **Chart fact:** Saturn occupies its own Capricorn in the seventh, remains in Capricorn in Navamsa, and receives directional strength in the seventh under Braha’s rules.  
  **Interpretation:** This supports persistence, discipline, organisational capacity, and authority. However, Saturn is only **0°11′ into Capricorn**: Braha’s warning about planets at the very beginning of a sign materially qualifies an otherwise impressive strength profile.

- **The Moon has a useful Navamsa foundation.**  
  **Chart fact:** The ascendant lord is waxing, though not near full, and occupies its own Cancer in Navamsa.  
  **Interpretation:** These factors provide some underlying resilience despite its difficult natal sixth-house placement. Braha’s *Navamsa* and *Planetary Strengths* sections explicitly recognise own-sign Navamsa as strengthening.

- **There are genuine wealth-producing connections.**  
  The second lord Sun joins the eleventh lord Venus; Mars and Jupiter both aspect the eleventh house. Braha’s concluding lordship combinations and Harihar’s *Dhana Bhava* discussion support financial potential through such links. Their effectiveness is reduced by the condition of Venus and Jupiter, so they do not establish effortless prosperity.

## 3. Principal unfavourable configurations

- **The ascendant lord occupies the sixth house.**  
  The Moon places considerable emphasis on work, service, competition, and overcoming difficulties, but is less favourable for ease and emotional tranquillity. Braha treats this as a vulnerability for general well-being; Harihar allows more favourable results when the ascendant lord is otherwise fortified. Its own-sign Navamsa is therefore an important qualification.

- **Jupiter is exceptionally close to its deepest debilitation.**  
  Jupiter stands at **4°47′ Capricorn**, close to the **5° deepest-fall point** specified by Harihar. It joins Saturn and receives Mars’s aspect. Its ninth-house promise is consequently compromised, notwithstanding cancellation factors.  
  The chart labels Jupiter a functional malefic; both supplied authors nevertheless recognise its ninth lordship as beneficial for Cancer ascendant while acknowledging its sixth-house complications. **“Mixed and afflicted” is more informative here than an unqualified benefic or malefic label.**

- **The fourth–seventh-house relationship is heavily pressured.**  
  Mars and Mercury occupy the fourth; Saturn aspects both, while Mars aspects Saturn and Jupiter in the seventh. This links home, contentment, education, property, partnership, and professional responsibilities in a demanding configuration. Harihar regards Mars–Saturn contact as afflicting both planets; Braha additionally associates their mutual aspect with technical ability. Thus **competence and strain can arise from the same configuration**.

- **The financial–shared-resource axis needs caution.**  
  Rahu occupies Leo in the second, while Ketu occupies the eighth. Braha specifically regards this nodal axis as disturbing family happiness and financial harmony, with Leo an unfriendly placement for Rahu. Jupiter’s aspect on the second lord offers mitigation, but Jupiter itself is impaired.

## 4. Important yogas, exchanges, cancellations, and exceptions

### Verified sign exchange: Mercury–Venus

**Mercury occupies Libra, Venus’s sign; Venus occupies Virgo, Mercury’s sign.** This exchanges the **third and fourth houses**, while linking Mercury’s additional twelfth lordship with Venus’s eleventh lordship.

Braha’s *Exchange of Signs* says mutual reception strengthens the participating planets, but his later qualification warns that adverse lordships matter. Accordingly, this is **not a clean prosperity-giving exchange**: communication, effort, home, gains, and expenditure become closely interconnected. Mercury’s retrogression and affliction, and Venus’s debilitation, prevent treating the exchange as an uncomplicated benefit.

### Neechabhanga: present for both debilitated planets

- **Jupiter:** Its dispositor Saturn occupies an angle in its own sign; Mars, the planet exalted in Capricorn, also occupies an angle.
- **Venus:** Its dispositor Mercury occupies an angle.

These satisfy cancellation conditions in Braha’s *Neechabhanga Yoga*. Harihar additionally supports cancellation where the dispositor occupies its own sign, directly reinforcing Jupiter’s case. **Cancellation is not complete restoration:** Braha expressly retains difficulties alongside possible benefits.

### Sasa yoga: formation present, effectiveness qualified

Saturn in its own sign in an angle meets Harihar’s stated **Sasa yoga** condition. Its Navamsa placement reinforces the formation, but its extremely early degree, eighth lordship, and Mars’s aspect qualify the outcome.

### Lunar yogas: mixed support

- **Sunapha yoga is present:** Jupiter and Saturn occupy the second from the Moon.
- **Sakata yoga is also present under Braha’s definition:** the Moon occupies the twelfth from Jupiter.
- **Kemadruma is absent:** the house following the Moon is occupied.
- **Gajakesari is absent:** Jupiter is not angular from the Moon.

This is a clear example of favourable and adverse indications coexisting rather than cancelling one another mechanically.

### Marriage-affliction rules disagree

Mars in the fourth in Libra meets Braha’s **Kujadosha** condition, without his sign exception. Harihar, however, supplies a neutralisation rule involving Saturn or Rahu in the listed marriage-sensitive houses; Saturn in the seventh supplies that mitigating condition here. Harihar also regards own-sign Saturn in the seventh more favourably than Braha, who retains substantial relationship difficulty even there. **A categorical adverse verdict would ignore these exceptions and disagreements.**

## 5. Overall synthesis

The strongest theme is **achievement through effort, discipline, and practical ability—not uncomplicated ease**. Mars provides direction; Saturn supplies endurance but also pressure. The Sun’s Virgo vargottama offers support, although its late natal degree qualifies its strength. Mercury is capable but retrograde and afflicted; Venus gains help through exchange and cancellation but remains compromised. Jupiter’s cancellation is important precisely because its underlying debilitation is so pronounced.

The chart supports neither a blanket promise of high prosperity nor a blanket judgement of deprivation. Its defining feature is the coexistence of **constructive capacity with substantial emotional, domestic, and partnership demands**. No single yoga adequately describes the whole horoscope.

## Add Gochar and Transit Info

In [30]:
Gochar = readText(GOCHAR)
Transits = readText(GOCHAR_EVENTS)

/content/Gochar_Chart.txt | 4,792 characters
/content/Transit.txt | 5,654 characters


In [32]:
# ------------------------------------------------
# Set the Context for Subsequent Passes
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)
'''
CONTEXT = getContext(
    RAMANYOG = RamanYog,
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA,
    FIRSTPASS=FirstPass,
    GOCHAR=Gochar,
    TRANSITS=Transits
)


'''
CONTEXT2 = augContext(
    CONTEXT,
    RAMANYOG = RamanYog,
    FIRSTPASS=FirstPass,
    GOCHAR=Gochar,
    TRANSITS=Transits
)


## Konsultation

In [33]:
# ------------------------------------------------
# Ask your question / define the task
# This and subsequent cells can be run multiple times
# ------------------------------------------------
CurrentQuestion = """
Give a detailed overview of the native's life over the next two years based on dasha, gochar, transits.
Focus on fame, recognition, wealth and relationship with family members
"""

In [39]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

#model = "gpt-6.1-sol"

QUERY = buildQuery(CONTEXT2, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6.1-sol | Effort: medium | In: 231,641 | Cached: 0 | Out: 5,559 | Reasoning: 1,034 | Time: 180.5s


In [40]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, Native_id, CurrentQuestion, provider,model)

Saved: /content/K21_261104_131851_gpt-6.1-sol.md
Saved: /content/K21_261104_131851_gpt-6.1-sol.docx


## Check Credit Status
https://platform.openai.com/settings/organization/billing/credit-grants

https://platform.openai.com/usage


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)